In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download purusinghvi/email-spam-classification-dataset

Dataset URL: https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset
License(s): MIT
  0% 0.00/43.0M [00:00<?, ?B/s]
100% 43.0M/43.0M [00:00<00:00, 1.36GB/s]


In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/email-spam-classification-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import pandas as pd
df=pd.read_csv("/content/email-spam-classification-dataset.zip")

In [ ]:
df.head()

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [ ]:
df['text']=df['text'].str.lower()

In [ ]:
import string
symbs=string.punctuation

In [ ]:
symbs

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
df['text']

,text
0,ounce feather bowl hummingbird opec moment ala...
1,wulvob get your medircations online qnb ikud v...
2,computer connection from cnn com wednesday es...
3,university degree obtain a prosperous future m...
4,thanks for all your answers guys i know i shou...
...,...
83443,hi given a date how do i get the last date of ...
83444,now you can order software on cd or download i...
83445,dear valued member canadianpharmacy provides a...
83446,subscribe change profile contact us long term ...


In [ ]:
def remove(txt):
  return txt.translate(str.maketrans('','',symbs))
df['text']=df['text'].apply(remove)

In [ ]:
df.head()

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
word_set=set(stopwords.words('english'))
def remove_stopwords(txt):
  return " ".join([word for word in txt.split() if word not in word_set])
df['text']=df['text'].map(remove_stopwords)

In [ ]:
import re
def remove_tags(text):
    pattern = re.compile(r'<.*?>')
    clean_text = pattern.sub('', text)
    return clean_text

In [ ]:
df['text']=df['text'].map(remove_tags)

In [ ]:
import re

def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U0001F900-\U0001F9FF"
        "\U00002600-\U000026FF"
        "\U00002B00-\U00002BFF"
        "]+", flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)


In [ ]:
df['text']=df['text'].map(remove_emoji)

In [ ]:
df['length']=df['text'].map(len)

In [ ]:
df.head()

,label,text,length
0,1,ounce feather bowl hummingbird opec moment ala...,148
1,1,wulvob get medircations online qnb ikud viagra...,716
2,0,computer connection cnn com wednesday escapenu...,1995
3,1,university degree obtain prosperous future mon...,510
4,0,thanks answers guys know checked rsync manual ...,1089


In [ ]:
df.head()

,label,text,length
0,1,ounce feather bowl hummingbird opec moment ala...,148
1,1,wulvob get medircations online qnb ikud viagra...,716
2,0,computer connection cnn com wednesday escapenu...,1995
3,1,university degree obtain prosperous future mon...,510
4,0,thanks answers guys know checked rsync manual ...,1089


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

In [ ]:
x=cv.fit_transform(df['text'])

In [ ]:
y=df['label']

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model=MultinomialNB()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9787297783103655
[[7854   84]
 [ 271 8481]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      7938
           1       0.99      0.97      0.98      8752

    accuracy                           0.98     16690
   macro avg       0.98      0.98      0.98     16690
weighted avg       0.98      0.98      0.98     16690



In [ ]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6426391 stored elements and shape (66758, 310795)>

In [ ]:
from sklearn.metrics import precision_score
print(precision_score(y_test,y_pred))

0.9901926444833625


In [ ]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(cv, "vectorizer.pkl")

['vectorizer.pkl']

In [ ]:
from google.colab import files
files.download("spam_model.pkl")
files.download("vectorizer.pkl")

In [ ]:
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>